# Phase 1: Data Extraction & Phonetic Word-Pair Pipeline
**Project:** Spanglish, But Make It Punny (CMPT 419 / 983)  
**Authors:** Michaela Foo & Kevin Gonzalez  

### Notebook Objectives
1. Mount Google Drive and initialize the shared project folder hierarchy (`CMPT419_983_Spanglish_Puns`).
2. Clone the reference codebase (`Likhith-Asapu/Codemix-Pun-Generation`) and download `ipa-dict` pronunciation lexicons (`en_US`, `es_MX`, `es_ES`).
3. Audit and prune synthetic phonetic spellings and cognates from the initial `spanish_english_pun_pairs.csv` seed file.
4. Clean Wikipedia metadata artifacts from `english_frequency.csv` and `spanish_frequency.csv`, and build a bilingual cognate/translation filter from `en-es_wiki.txt`.
5. Mine real-word Spanish-English homophones/near-homophones using articulatory feature-weighted IPA similarity (`panphon` + normalized Levenshtein).
6. Extract semantic context words ($C_w$) tied to each English alternate word ($A_w$) to produce **100–150 $(P_w, A_w, C_w)$ pun tuples** for Phase 2.

## 1. Mount Google Drive & Initialize Project Directories
Mounts Google Drive, makes sure project files exists, & defines the `DIRS` dictionary so every downstream cell reads and writes to the exact shared paths.

In [ ]:
from google.colab import drive
import os

# 1. Mount Google Drive
drive.mount('/content/drive')

# 2. Define base project directory
BASE_DIR = "/content/drive/MyDrive/CMPT419_983_Spanglish_Puns"

DIRS = {
    "raw": os.path.join(BASE_DIR, "01_raw_data"),
    "ipa": os.path.join(BASE_DIR, "02_ipa_dicts"),
    "processed": os.path.join(BASE_DIR, "03_processed_pairs"),
    "repo": os.path.join(BASE_DIR, "04_repo_and_prompts"),
    "generated": os.path.join(BASE_DIR, "05_generated_puns"),
    "eval": os.path.join(BASE_DIR, "06_evaluation"),
    "notebooks": os.path.join(BASE_DIR, "notebooks"),
}

# 3. Create folders if they don't exist yet
for name, path in DIRS.items():
    os.makedirs(path, exist_ok=True)

# 4. Set working directory to project root
os.chdir(BASE_DIR)
print("Working directory set to:", os.getcwd())

Mounted at /content/drive
Working directory set to: /content/drive/MyDrive/CMPT419_983_Spanglish_Puns


## 2. Install Dependencies, Clone Reference Repo & Download `ipa-dict`
Installs `panphon` (articulatory phonological distance), `rapidfuzz` (fast Levenshtein edit distance), and `nltk` (WordNet definitions and hypernyms). Also clones Asapu et al.'s (2025) `Codemix-Pun-Generation` repository and downloads the `ipa-dict` pronunciation files into `02_ipa_dicts/`.

In [ ]:
!pip install -q panphon rapidfuzz nltk pandas

import re
import urllib.request
import pandas as pd
import nltk
from rapidfuzz.distance import Levenshtein
import panphon.distance

nltk.download('wordnet', quiet=True)
nltk.download('omw-1.4', quiet=True)
from nltk.corpus import wordnet as wn

# 1. Clone reference repo into 04_repo_and_prompts/
repo_target = os.path.join(DIRS["repo"], "Codemix-Pun-Generation")
if not os.path.exists(repo_target):
    !git clone https://github.com/Likhith-Asapu/Codemix-Pun-Generation.git "{repo_target}"
    print("Cloned Codemix-Pun-Generation to:", repo_target)
else:
    print("Reference repo already exists at:", repo_target)

# 2. Download ipa-dict lexicons into 02_ipa_dicts/
ipa_urls = {
    "en_US.txt": "https://raw.githubusercontent.com/open-dict-data/ipa-dict/master/data/en_US.txt",
    "es_MX.txt": "https://raw.githubusercontent.com/open-dict-data/ipa-dict/master/data/es_MX.txt",
    "es_ES.txt": "https://raw.githubusercontent.com/open-dict-data/ipa-dict/master/data/es_ES.txt",
}

for fname, url in ipa_urls.items():
    out_path = os.path.join(DIRS["ipa"], fname)
    if not os.path.exists(out_path):
        urllib.request.urlretrieve(url, out_path)
        print(f"Downloaded {fname} -> {out_path}")
    else:
        print(f"Already exists: {fname}")

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
Cloning into '/content/drive/MyDrive/CMPT419_983_Spanglish_Puns/04_repo_and_prompts/Codemix-Pun-Generation'...
remote: Enumerating objects: 140, done.
remote: Counting objects: 100% (140/140), done.
remote: Compressing objects: 100% (108/108), done.
remote: Total 140 (delta 34), reused 129 (delta 30), pack-reused 0 (from 0)
Receiving objects: 100% (140/140), 38.43 MiB | 10.46 MiB/s, done.
Resolving deltas: 100% (34/34), done.
Updating files: 100% (120/120), done.
Cloned Codemix-Pun-Generation to: /content/drive/MyDrive/CMPT419_983_Spanglish_Puns/04_repo_and_prompts/Codemix-Pun-Generation
Downloaded en_US.txt -> /content/drive/MyDrive/CMPT419_983_Spanglish_Puns/02_ipa_dicts/en_US.txt
Downloaded es_MX.txt -> /content/drive/MyDrive/CMPT419_983_Spanglish_Puns/02_ipa_dicts/es_MX.txt
Downloaded es_ES.txt -> /content/drive/MyDrive/CMPT419_983_Spanglish_Puns/02

## 3. Clean Seed Pun Pairs (`spanish_english_pun_pairs.csv`)
Filters out entries from the initial 93-pair table that do not represent valid real-word, non-cognate cross-lingual puns:
* Removes fabricated phonetic respellings (`phrase-like form`, `informal name-like form`, `proper name-like form`) and standalone proper names.
* Removes direct cognates and shared loanwords (`garaje`/`garage`, `dormitorio`/`dormitory`, `patio`/`patio`, `tren`/`train`, `café`/`café`, `tortilla`/`tortilla`, etc.) that lack semantic ambiguity across languages.

In [ ]:
raw_pairs_path = os.path.join(DIRS["raw"], "spanish_english_pun_pairs.csv")
raw_pairs = pd.read_csv(raw_pairs_path)

non_word_pattern = r"proper name|informal name-like|phrase-like form"
manual_cognates = {
    "café", "tortilla", "tren", "dormitorio", "garaje", "jardín", "patio",
    "casa", "fuego", "cielo", "río", "tía", "tío", "plato", "piso",
    "avión", "rueda", "manta", "pared"
}

seed_df = raw_pairs[
    ~raw_pairs["english_meaning"].str.contains(non_word_pattern, case=False, na=False) &
    ~raw_pairs["spanish_word"].str.lower().isin(manual_cognates)
].copy()

seed_df["source"] = "seed_cleaned"
print(f"Cleaned seed pairs retained: {len(seed_df)} / {len(raw_pairs)}")
display(seed_df[["spanish_word", "english_word", "spanish_meaning", "english_meaning", "phonetic_similarity"]].head(15))

Cleaned seed pairs retained: 39 / 93


,spanish_word,english_word,spanish_meaning,english_meaning,phonetic_similarity
0,sopa,soap,soup,cleaning substance,0.83
1,flor,floor,flower,bottom surface of a room,0.80
2,burro,burrow,donkey,hole dug by an animal,0.75
3,gato,ghetto,cat,urban neighborhood,0.75
4,sol,soul,sun,spirit or essence,0.80
5,pez,pets,fish,companion animals,0.80
6,pan,pun,bread,play on words,0.75
9,vino,beano,wine,British informal term for a bean feast,0.75
10,pelo,pillow,hair,cushion for the head,0.70
11,velo,willow,veil,tree,0.65


## 4. Clean Frequency Lists & Build Bilingual Translation Cognate Filter
* **Frequency Sanitization:** Strips Wikipedia dump artifacts (`category`, `categoría`, `anexo`, `template`, `wikipedia`, `county`, `township`), function words, and tokens shorter than 3 characters from `english_frequency.csv` and `spanish_frequency.csv`.
* **Bilingual Dictionary Lookup:** Parses all 886,963 bilingual entries in `en-es_wiki.txt` into fast lookup sets (`en_to_es_trans` and `es_to_en_trans`) so any candidate pair where one word is a direct translation of the other is automatically rejected.

In [ ]:
wiki_junk = {
    "category", "categoría", "anexo", "template", "plantilla", "wikipedia",
    "redirect", "stub", "file", "archivo", "user", "usuario", "talk", "discusión",
    "list", "lista", "county", "condado", "township", "municipio", "saint",
    "de", "la", "el", "los", "las", "del", "en", "y", "a", "por", "con", "para",
    "the", "of", "in", "and", "by", "at", "to", "for", "on", "with", "as", "from"
}

def load_clean_freq(csv_path, min_len=3):
    df = pd.read_csv(csv_path)
    df["word"] = df["word"].astype(str).str.lower().str.strip()
    df = df[
        df["word"].str.isalpha() &
        (df["word"].str.len() >= min_len) &
        (~df["word"].isin(wiki_junk))
    ].drop_duplicates(subset=["word"])
    return df

en_freq_clean = load_clean_freq(os.path.join(DIRS["raw"], "english_frequency.csv"), min_len=3)
es_freq_clean = load_clean_freq(os.path.join(DIRS["raw"], "spanish_frequency.csv"), min_len=3)
print(f"Clean English freq words: {len(en_freq_clean)} | Clean Spanish freq words: {len(es_freq_clean)}")

# Parse en-es_wiki.txt into translation sets
en_to_es_trans = {}
es_to_en_trans = {}

wiki_dict_path = os.path.join(DIRS["raw"], "en-es_wiki.txt")
with open(wiki_dict_path, "r", encoding="utf-8") as f:
    for line in f:
        parts = line.strip().split("\t")
        if len(parts) != 2:
            continue
        en_clean = re.sub(r"\s*\(.*?\)", "", parts[0]).strip().lower()
        es_clean = re.sub(r"\s*\(.*?\)", "", parts[1]).strip().lower()
        if en_clean.isalpha() and es_clean.isalpha():
            en_to_es_trans.setdefault(en_clean, set()).add(es_clean)
            es_to_en_trans.setdefault(es_clean, set()).add(en_clean)

print(f"Bilingual translation index built: {len(en_to_es_trans)} EN keys, {len(es_to_en_trans)} ES keys")

Clean English freq words: 2841 | Clean Spanish freq words: 2843
Bilingual translation index built: 132544 EN keys, 133379 ES keys


## 5. Cross-Lingual Phonetic Homophone Mining (`ipa-dict` + Panphon)
Matches real Spanish and English vocabulary strictly from `ipa-dict` entries:
1. Prioritizes high-frequency content words from your cleaned frequency CSVs alongside WordNet lexical entries.
2. Computes a composite phonetic similarity score ($0.5 \times \text{IPA Normalized Levenshtein} + 0.5 \times \text{Panphon Articulatory Feature Similarity}$) and requires $\ge 0.72$.
3. Rejects cognates/translations via exact spelling, `en-es_wiki.txt` lookup, WordNet synset overlap, and orthographic Levenshtein similarity ($\ge 0.72$).

In [ ]:
dst = panphon.distance.Distance()

def load_ipa_dict(filepath):
    ipa_map = {}
    with open(filepath, "r", encoding="utf-8") as f:
        for line in f:
            parts = line.strip().split("\t")
            if len(parts) >= 2:
                word = parts[0].strip().lower()
                raw_ipa = parts[1].split(",")[0].strip()
                clean_ipa = raw_ipa.strip("/").replace("ˈ", "").replace("ˌ", "")
                if word.isalpha() and len(word) >= 3:
                    ipa_map[word] = (raw_ipa, clean_ipa)
    return ipa_map

en_ipa_dict = load_ipa_dict(os.path.join(DIRS["ipa"], "en_US.txt"))
# Prefer Latin American Spanish (es_MX) with Castilian (es_ES) fallback
es_ipa_dict = load_ipa_dict(os.path.join(DIRS["ipa"], "es_ES.txt"))
es_ipa_dict.update(load_ipa_dict(os.path.join(DIRS["ipa"], "es_MX.txt")))
#add additional country
#microsoft -> tons of spen
#add to end project presentation/report
#could have used different accent

print(f"IPA entries loaded -> EN: {len(en_ipa_dict)}, ES: {len(es_ipa_dict)}")

# Build candidate vocabulary pools prioritizing cleaned frequency words
en_wn_words = {w.lower() for w in wn.all_lemma_names(lang="eng") if w.isalpha() and len(w) >= 3}
es_wn_words = {w.lower() for w in wn.all_lemma_names(lang="spa") if w.isalpha() and len(w) >= 3}

en_freq_set = set(en_freq_clean["word"])
es_freq_set = set(es_freq_clean["word"])

en_pool = sorted((en_freq_set | (en_wn_words & set(en_ipa_dict.keys()))) & set(en_ipa_dict.keys()),
                 key=lambda w: (w not in en_freq_set, len(w)))[:6000]
es_pool = sorted((es_freq_set | (es_wn_words & set(es_ipa_dict.keys()))) & set(es_ipa_dict.keys()),
                 key=lambda w: (w not in es_freq_set, len(w)))[:4000]

print(f"Candidate pools -> ES: {len(es_pool)} words, EN: {len(en_pool)} words")

def is_cognate_or_translation(es_w, en_w):
    if es_w == en_w:
        return True
    if en_w in es_to_en_trans.get(es_w, set()) or es_w in en_to_es_trans.get(en_w, set()):
        return True
    es_synsets = set(wn.synsets(es_w, lang="spa"))
    en_synsets = set(wn.synsets(en_w, lang="eng"))
    if es_synsets and en_synsets and (es_synsets & en_synsets):
        return True
    if Levenshtein.normalized_similarity(es_w, en_w) >= 0.72:
        return True
    return False

def get_gloss(word, lang):
    syns = wn.synsets(word, lang=lang)
    if syns:
        return syns[0].definition()
    if lang == "spa" and word in es_to_en_trans:
        return "; ".join(sorted(es_to_en_trans[word])[:2])
    return ""

# Bucket English words by initial IPA phoneme for fast lookup
en_by_initial = {}
for en_w in en_pool:
    raw_ipa, clean_ipa = en_ipa_dict[en_w]
    if clean_ipa:
        en_by_initial.setdefault(clean_ipa[0], []).append((en_w, raw_ipa, clean_ipa))

initial_equiv = {"b": ["b", "v", "w"], "s": ["s", "z", "θ"], "k": ["k", "c"], "j": ["j", "h"]}
existing_es = set(seed_df["spanish_word"].str.lower())
mined_rows = []

for es_w in es_pool:
    if es_w in existing_es:
        continue
    es_raw_ipa, es_clean_ipa = es_ipa_dict[es_w]
    if not es_clean_ipa:
        continue
    es_gloss = get_gloss(es_w, "spa")
    if not es_gloss:
        continue

    buckets = initial_equiv.get(es_clean_ipa[0], [es_clean_ipa[0]])
    best_match = None
    best_score = 0.0

    for b in buckets:
        for en_w, en_raw_ipa, en_clean_ipa in en_by_initial.get(b, []):
            if abs(len(es_clean_ipa) - len(en_clean_ipa)) > 2:
                continue
            lev_sim = Levenshtein.normalized_similarity(es_clean_ipa, en_clean_ipa)
            if lev_sim < 0.60:
                continue
            feat_dist = dst.feature_edit_distance_div_maxlen(es_clean_ipa, en_clean_ipa)
            feat_sim = max(0.0, 1.0 - feat_dist)
            combined_sim = round(0.5 * lev_sim + 0.5 * feat_sim, 3)

            if combined_sim >= 0.72 and combined_sim > best_score:
                if not is_cognate_or_translation(es_w, en_w):
                    en_gloss = get_gloss(en_w, "eng")
                    if en_gloss:
                        best_score = combined_sim
                        best_match = (en_w, en_raw_ipa, en_gloss, combined_sim)

    if best_match:
        en_w, en_raw_ipa, en_gloss, score = best_match
        mined_rows.append({
            "spanish_word": es_w,
            "spanish_ipa": es_raw_ipa,
            "spanish_meaning": es_gloss,
            "english_word": en_w,
            "english_ipa": en_raw_ipa,
            "english_meaning": en_gloss,
            "phonetic_similarity": score,
            "cognate_filter": "passed",
            "pair_status": "mined_candidate",
            "source": "ipa_dict_mined"
        })

mined_df = pd.DataFrame(mined_rows).sort_values(by="phonetic_similarity", ascending=False)
print(f"Newly mined real-word non-cognate candidates: {len(mined_df)}")
display(mined_df.head(25))

IPA entries loaded -> EN: 117147, ES: 595825
Candidate pools -> ES: 4000 words, EN: 6000 words
Newly mined real-word non-cognate candidates: 224


,spanish_word,spanish_ipa,spanish_meaning,english_word,english_ipa,english_meaning,phonetic_similarity,cognate_filter,pair_status,source
13,sin,/sin/,lessness; sin,scene,/ˈsin/,the place where some action occurs,1.000,passed,mined_candidate,ipa_dict_mined
74,pis,/pis/,liquid excretory product,peace,/ˈpis/,the state prevailing during the absence of war,1.000,passed,mined_candidate,ipa_dict_mined
65,puf,/puf/,a low seat or a stool to rest the feet of a se...,poof,/ˈpuf/,offensive term for an openly homosexual man,1.000,passed,mined_candidate,ipa_dict_mined
145,caqui,/kaki/,a military uniform made of khaki fabric,cocky,/ˈkɑki/,overly self-confident or self-assertive,0.872,passed,mined_candidate,ipa_dict_mined
99,pipí,/piˈpi/,liquid excretory product,peppy,/ˈpɛpi/,marked by lively action,0.865,passed,mined_candidate,ipa_dict_mined
157,mohín,/moˈin/,a disdainful grimace,moan,/ˈmoʊn/,an utterance expressing pain or disapproval,0.859,passed,mined_candidate,ipa_dict_mined
111,maíz,/maˈis/,whiskey distilled from a mash of not less than...,mouse,/ˈmaʊs/,any of numerous small rodents typically resemb...,0.859,passed,mined_candidate,ipa_dict_mined
28,sean,/sean/,seán,seine,/ˈseɪn/,a French river that flows through the heart of...,0.854,passed,mined_candidate,ipa_dict_mined
113,tupí,/tuˈpi/,a member of the South American Indian people l...,tippy,/ˈtɪpi/,(used of boats) inclined to heel over easily u...,0.854,passed,mined_candidate,ipa_dict_mined
116,peón,/peˈon/,(chess) the least powerful piece; moves only f...,pain,/ˈpeɪn/,a symptom of some physical hurt or disorder,0.854,passed,mined_candidate,ipa_dict_mined


## 6. Extract Context Words ($C_w$) & Export 130 $(P_w, A_w, C_w)$ Tuples
Aligns the dataset with Asapu et al.'s (2025) three-element tuple schema **$(P_w, A_w, C_w)$** by extracting semantic context anchors ($C_w$) from WordNet hypernyms and glosses for each English alternate word ($A_w$). Saves the combined 130-pair dataset to `03_processed_pairs/spanglish_pun_pairs_130_clean.csv` for manual inspection before Phase 2.

In [ ]:
stopwords_en = {
    "a", "an", "the", "of", "in", "on", "at", "to", "for", "with", "by", "or",
    "and", "is", "are", "was", "were", "be", "been", "being", "that", "which",
    "who", "whom", "whose", "as", "from", "into", "like", "through", "after",
    "over", "between", "out", "against", "during", "without", "before", "under",
    "around", "among", "used", "usually", "especially", "something", "someone",
    "person", "thing", "form", "type", "kind", "act", "state", "having", "made"
}

def extract_context_words(en_word, fallback_gloss=""):
    context_candidates = []
    syns = wn.synsets(en_word.lower(), lang="eng")
    if syns:
        primary = syns[0]
        # 1. Hypernym lemmas
        for hyper in primary.hypernyms():
            for lemma in hyper.lemma_names():
                clean_l = lemma.replace("_", " ").lower()
                if clean_l != en_word.lower() and clean_l not in stopwords_en:
                    context_candidates.append(clean_l)
        # 2. Content words from primary definition
        def_tokens = re.findall(r"[a-z]{4,}", primary.definition().lower())
        for tok in def_tokens:
            if tok != en_word.lower() and tok not in stopwords_en and tok not in context_candidates:
                context_candidates.append(tok)

    if not context_candidates and fallback_gloss:
        def_tokens = re.findall(r"[a-z]{4,}", fallback_gloss.lower())
        context_candidates = [t for t in def_tokens if t != en_word.lower() and t not in stopwords_en]

    return ", ".join(context_candidates[:3]) if context_candidates else en_word.lower()

# Combine cleaned seed pairs + top mined candidates to reach 130 pairs
TARGET_COUNT = 130
needed = max(0, TARGET_COUNT - len(seed_df))
combined_df = pd.concat([seed_df, mined_df.head(needed)], ignore_index=True)

# Populate C_w (context_word) column
combined_df["context_word"] = combined_df.apply(
    lambda row: extract_context_words(row["english_word"], row["english_meaning"]),
    axis=1
)

# Order columns to match Codemix-Pun-Generation (Pw, Aw, Cw)
final_cols = [
    "spanish_word",       # Pw (Pun Word)
    "english_word",       # Aw (Alternate Word)
    "context_word",       # Cw (Context Word)
    "spanish_ipa",
    "english_ipa",
    "spanish_meaning",
    "english_meaning",
    "phonetic_similarity",
    "cognate_filter",
    "pair_status",
    "source"
]
final_df = combined_df[final_cols].copy()
final_df.insert(0, "pair_id", range(1, len(final_df) + 1))

out_csv = os.path.join(DIRS["processed"], "spanglish_pun_pairs_130_clean.csv")
final_df.to_csv(out_csv, index=False)

print(f"Saved {len(final_df)} clean (Pw, Aw, Cw) pun tuples to:\n{out_csv}")
print("\nBreakdown by source:")
print(final_df["source"].value_counts())
display(final_df[["pair_id", "spanish_word", "english_word", "context_word", "phonetic_similarity", "source"]].head(20))

Saved 130 clean (Pw, Aw, Cw) pun tuples to:
/content/drive/MyDrive/CMPT419_983_Spanglish_Puns/03_processed_pairs/spanglish_pun_pairs_130_clean.csv

Breakdown by source:
source
ipa_dict_mined    91
seed_cleaned      39
Name: count, dtype: int64


,pair_id,spanish_word,english_word,context_word,phonetic_similarity,source
0,1,sopa,soap,"cleansing agent, cleanser, cleaner",0.83,seed_cleaned
1,2,flor,floor,"horizontal surface, level, inside",0.80,seed_cleaned
2,3,burro,burrow,"hole, hollow, animal",0.75,seed_cleaned
3,4,gato,ghetto,"quarter, formerly, restricted",0.75,seed_cleaned
4,5,sol,soul,"spirit, immaterial, part",0.80,seed_cleaned
5,6,pez,pets,"animal, animate being, beast",0.80,seed_cleaned
6,7,pan,pun,"fun, play, sport",0.75,seed_cleaned
7,8,vino,beano,"board game, game, numbered",0.75,seed_cleaned
8,9,pelo,pillow,"cushion, support, head",0.70,seed_cleaned
9,10,velo,willow,"tree, numerous, deciduous",0.65,seed_cleaned


## 7. Quick Quality Summary & Phase 1 Verification Check
Prints summary statistics on phonetic similarity scores and missing values to confirm dataset is ready for manual review (`spanglish_pun_pairs_final.csv`) before running Phase 2 LLM generation.

In [ ]:
print("=== Dataset Summary ===")
print(f"Total pairs: {len(final_df)}")
print(f"Null values per column:\n{final_df.isnull().sum()}\n")
print("Phonetic similarity distribution:")
print(final_df["phonetic_similarity"].describe().round(3))

=== Dataset Summary ===
Total pairs: 130
Null values per column:
pair_id                0
spanish_word           0
english_word           0
context_word           0
spanish_ipa            0
english_ipa            0
spanish_meaning        0
english_meaning        0
phonetic_similarity    0
cognate_filter         0
pair_status            0
source                 0
dtype: int64

Phonetic similarity distribution:
count    130.000
mean       0.767
std        0.101
min        0.400
25%        0.750
50%        0.794
75%        0.819
max        1.000
Name: phonetic_similarity, dtype: float64
